In [20]:
%pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 502.6 kB/s  0:00:04494.7 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [yfinance]━━ 3/5 [curl_cffi]
Note: you may need to restart the kernel to use updated packages.


In [21]:
# Celda B1 - Configuracion y universo a enriquecer
# En terminal, si falta la libreria: pip install yfinance
import yfinance as yf
import pandas as pd
import numpy as np
import time
from pathlib import Path
from datetime import date

HOY = date.today().isoformat()
import os
V2 = Path(os.environ.get('TESIS_BASE', '/Users/ppizam/Claude/Master Thesis')) / 'Desarrollo/Metodologia/Lista Maestra de Tickers/Lista maestra V2'

vig = pd.read_csv(V2 / 'padron_vigencias_2020_2024_mkt_flags.csv',
                  keep_default_na=False, na_values=[''])
print(len(vig), vig['permno'].nunique())   # debe dar 12961 11899

# Universo: vidas activas cuyo ticker SIGUE listado hoy (las que cruzaron en la fase A).
# El archivo de campos del directorio se busca por patron para no depender de la fecha:
campos_dir_path = sorted(V2.glob('campos_directorio_*.csv'))[-1]
print('usando:', campos_dir_path.name)
campos_dir = pd.read_csv(campos_dir_path)

llaves_listados = set(zip(campos_dir['permno'], campos_dir['bloque']))
universo = vig[[tuple(x) in llaves_listados for x in zip(vig['permno'], vig['bloque'])]]
tickers = sorted(universo['ticker'].unique())
print(len(tickers), 'tickers a enriquecer')   # esperado: ~7,000-7,800

12961 11899
usando: campos_directorio_2026-07-17.csv
7760 tickers a enriquecer


In [22]:
# Celda B2 - Prueba con un ticker y verificacion de campos disponibles
# Los nombres de campos de yfinance cambian entre versiones: verificar ANTES del loop
t = yf.Ticker('GME')
info = t.info
CAMPOS = ['floatShares', 'shortPercentOfFloat', 'sharesShort', 'averageVolume',
          'averageVolume10days', 'country', 'industry', 'sector', 'currentPrice']
for c in CAMPOS:
    print(f'{c:22}', info.get(c))
# Todos deben traer valor para GME. Si alguno sale None, revisar la salida
# antes de correr el loop completo

floatShares            408807091
shortPercentOfFloat    0.1364
sharesShort            55856110
averageVolume          6823698
averageVolume10days    3137880
country                United States
industry               Specialty Retail
sector                 Consumer Cyclical
currentPrice           21.89


In [31]:
# Celda B3b - Preparar reintento de los tickers con error
import pandas as pd
from pathlib import Path

import os
V2 = Path(os.environ.get('TESIS_BASE', '/Users/ppizam/Claude/Master Thesis')) / 'Desarrollo/Metodologia/Lista Maestra de Tickers/Lista maestra V2'
CHECKPOINT = V2 / 'enriquecimiento_yahoo_2026-07-17.csv'

prev = pd.read_csv(CHECKPOINT)
antes = (prev['error'].fillna('') != '').sum()
prev = prev[prev['error'].fillna('') == '']
prev.to_csv(CHECKPOINT, index=False)
print('listos para reintento:', antes, 'tickers; ahora corre la celda B3')

listos para reintento: 0 tickers; ahora corre la celda B3


In [32]:
# Celda B3 - Loop de enriquecimiento con checkpoint (LENTO: ~1.5 horas; dejar corriendo)
CHECKPOINT = V2 / 'enriquecimiento_yahoo_2026-07-17.csv'   # fijo: el archivo del viernes
PAUSA = 0.6          # segundos entre tickers; subir a 1.5 si Yahoo empieza a fallar
CADA = 100           # guardar checkpoint cada N tickers

hechos = set()
filas = []
if CHECKPOINT.exists():
    prev = pd.read_csv(CHECKPOINT)
    hechos = set(prev['ticker'])
    filas = prev.to_dict('records')
    print('reanudando:', len(hechos), 'ya hechos')

pendientes = [t for t in tickers if t not in hechos]
print(len(pendientes), 'pendientes')

for i, tk in enumerate(pendientes, 1):
    fila = {'ticker': tk, 'snapshot_yahoo': HOY, 'error': ''}
    try:
        info = yf.Ticker(tk).info
        fila.update({
            'float_shares': info.get('floatShares'),
            'short_percent_float': info.get('shortPercentOfFloat'),
            'shares_short': info.get('sharesShort'),
            'avg_volume_3m': info.get('averageVolume'),
            'avg_volume_10d': info.get('averageVolume10days'),
            'country': info.get('country'),
            'industry': info.get('industry'),
            'sector_yahoo': info.get('sector'),
            'price_yahoo': info.get('currentPrice'),
        })
    except Exception as e:
        fila['error'] = str(e)[:120]
    filas.append(fila)

    if i % CADA == 0 or i == len(pendientes):
        pd.DataFrame(filas).to_csv(CHECKPOINT, index=False)
        ok = sum(1 for f in filas if not f.get('error'))
        print(f'{i}/{len(pendientes)} | ok acumulado: {ok} | guardado')
    time.sleep(PAUSA)

print('terminado:', len(filas), 'tickers en', CHECKPOINT.name)

reanudando: 7760 ya hechos
0 pendientes
terminado: 7760 tickers en enriquecimiento_yahoo_2026-07-17.csv


In [33]:
# Celda B4 - Diagnostico al terminar (correr solo cuando B3 acabe)
res = pd.read_csv(CHECKPOINT)
print('total:', len(res))
print('con error:', (res['error'].fillna('') != '').sum())
for c in ['float_shares', 'short_percent_float', 'avg_volume_3m', 'country', 'industry']:
    print(f'cobertura {c:22}', round(res[c].notna().mean(), 3))
# La tasa de faltantes se documenta; NO se rellena.
# Si los errores son masivos (>10%), probablemente es rate limit: sube PAUSA a 1.5
# en la celda B3 y re-correla (reanuda sola desde el checkpoint)

total: 7760
con error: 0
cobertura float_shares           0.586
cobertura short_percent_float    0.602
cobertura avg_volume_3m          0.999
cobertura country                0.636
cobertura industry               0.636


In [34]:
import shutil
shutil.copy(CHECKPOINT, V2 / 'enriquecimiento_yahoo_final.csv')
print('copiado a enriquecimiento_yahoo_final.csv')

copiado a enriquecimiento_yahoo_final.csv
